# HW7 Lab: Build a RAG System 📚🔍🤖

<!-- lab-header:begin -->
**MIS 769 · Big Data Analytics for Business · Fall 2026**

**Points:** 40  |  **Due:** Sunday, October 25, 2026 at 10:00 PM Pacific

**Dr. Richard J. Young** · UNLV Lee Business School
<!-- lab-header:end -->

> **Why are we learning this?** LLMs know the public internet up to their training
> cutoff — they know *nothing* about YOUR company's return policy, YOUR contracts, or
> last week's meeting notes. RAG (Retrieval-Augmented Generation) fixes that: retrieve
> the relevant documents, paste them into the prompt, let the model answer from them.
> It is the single most deployed LLM pattern in business today. The transferable skill:
> **connecting an LLM to private knowledge without retraining anything.**

**How this lab works:** everything below is a *working demo* — hit Runtime → Run all
and a complete RAG system builds and answers questions on a fictional company's
documents. Your job: (1) understand each stage, (2) twist the knobs in the 🧪
Experiment cells, and (3) rebuild the corpus from **a domain you're passionate about**
in the 🎮 Playground — where YOU can judge whether the answers are right.

**The pipeline:** Query → Retrieve context (vector search) → Augment prompt → Generate

```
Query ──▶ Embed ──▶ Search vector store ──▶ top-k chunks ──▶ "Context: … Question: …" ──▶ LLM ──▶ Answer
```

⚠️ **Do NOT use pre-built services like Google NotebookLM for this assignment.** The
point is to BUILD the pipeline yourself so you understand every moving part.


## Learning objectives

- Split long documents into overlapping chunks and reason about the size/overlap trade-off
- Embed documents and build a FAISS vector index for similarity search
- Implement retrieval (`retrieve_similar`) and a full RAG pipeline (`rag_answer`)
- Compare RAG answers vs. pure-LLM answers and explain when RAG wins


## Setup

Three models/libraries, all free and local: **sentence-transformers** to embed text,
**FAISS** to search vectors fast, and **flan-t5-base** (an *instruction-tuned* model —
it actually follows "answer the question from this context") to generate answers.


In [1]:
!pip install -q sentence-transformers faiss-cpu transformers

import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")   # 384-dim embeddings

# flan-t5 is a seq2seq (encoder-decoder) model. transformers v5 removed the
# "text2text-generation" pipeline task, and routing T5 through "text-generation"
# runs but returns garbage, so we call the seq2seq classes directly.
_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
_mod = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generator(prompt, max_new_tokens=80, **params):
    """Same shape the text2text-generation pipeline returned: [{"generated_text": ...}]."""
    ids = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
    out = _mod.generate(**ids, max_new_tokens=max_new_tokens, **params)
    return [{"generated_text": _tok.decode(out[0], skip_special_tokens=True)}]

print("✅ Embedding model + generator loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Embedding model + generator loaded


## Part 1: Document Chunking (6 pts)

Real documents are too long to embed whole: one embedding for a 5-page policy averages
away every detail, and stuffing whole documents into the prompt blows the context
window. So we **chunk**: split into pieces small enough to embed precisely, with
**overlap** so a sentence that straddles a boundary isn't cut in half and lost.

- **Small chunks** → precise retrieval, but each hit carries little context.
- **Large chunks** → more context per hit, but retrieval gets fuzzy and you waste prompt space.
- **Overlap** → insurance against boundary cuts; costs some storage duplication.

Here's a longer policy document to practice on:


In [2]:
policy_document = """TechGadgets Inc. Customer Policy Handbook.
Returns and Refunds. Customers may return most items within 30 days of delivery for a
full refund. Electronics must be returned within 14 days and must include all original
packaging and accessories. Refunds are issued to the original payment method within 5-7
business days after we receive the item. Opened software, gift cards, and clearance
items marked final sale cannot be returned. To start a return, log into your account
and select the order, or contact support for a prepaid shipping label.
Shipping. Standard shipping is free on orders over 50 dollars within the United States
and takes 3-5 business days. Express shipping costs 12.99 dollars and arrives in 1-2
business days. We ship internationally to over 50 countries; international orders may
be subject to customs duties which are the customer's responsibility. Orders placed
before 2pm Pacific ship the same business day.
Warranty. All electronics carry a 2-year manufacturer warranty covering defects in
materials and workmanship. The warranty does not cover accidental damage, water damage,
or normal wear. Extended protection plans are available at checkout for 1 to 3
additional years. Warranty claims require proof of purchase.
Accounts and Discounts. Students receive a 20 percent discount with a valid student ID
verified through our partner service. Premium members pay 29.99 dollars per month and
receive priority support, early access to sales, and free express shipping. Enterprise
customers receive dedicated account managers and volume pricing on orders over 100
units. Loyalty points are earned at 1 point per dollar and never expire."""

print(f"Document length: {len(policy_document.split())} words")

Document length: 256 words


`chunk_text` below is the workhorse: word-based chunks of `chunk_size` words,
where consecutive chunks share `overlap` words. Every RAG framework (LangChain,
LlamaIndex…) ships a fancier version of exactly this function.


In [3]:
def chunk_text(text, chunk_size=60, overlap=15):
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks

chunks = chunk_text(policy_document, chunk_size=60, overlap=15)
print(f"{len(chunks)} chunks of ~60 words (overlap 15)")
print("\nFirst chunk:\n", chunks[0])

6 chunks of ~60 words (overlap 15)

First chunk:
 TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Customers may return most items within 30 days of delivery for a full refund. Electronics must be returned within 14 days and must include all original packaging and accessories. Refunds are issued to the original payment method within 5-7 business days after we receive the item. Opened software, gift cards, and clearance


In [4]:
# Feel the trade-off: try three settings and eyeball what a 'chunk' contains.
for size, ov in [(30, 5), (60, 15), (150, 30)]:
    cs = chunk_text(policy_document, chunk_size=size, overlap=ov)
    print(f"chunk_size={size:3d} overlap={ov:2d} → {len(cs):2d} chunks | "
          f"e.g. '{cs[len(cs)//2][:70]}...'")

chunk_size= 30 overlap= 5 → 11 chunks | e.g. 'orders may be subject to customs duties which are the customer's respo...'
chunk_size= 60 overlap=15 →  6 chunks | e.g. 'customer's responsibility. Orders placed before 2pm Pacific ship the s...'
chunk_size=150 overlap=30 →  2 chunks | e.g. 'to over 50 countries; international orders may be subject to customs d...'


### 🧪 Experiment

- Set `overlap=0` and print the chunk boundaries — find a sentence that got cut in half. That lost sentence is a question your RAG system can no longer answer.
- Try `chunk_size=30, overlap=5` and read a middle chunk out loud: could it answer "what does the warranty NOT cover?" on its own? Keep this intuition — you'll watch it break retrieval in Part 4.
- Push overlap to `chunk_size // 2`. How many chunks now? Overlap is insurance, and this is its storage premium.


**In a markdown cell:** which setting would you pick for this document, and why?
(You'll defend this choice in Q1. Hint: could a 30-word chunk answer "what does the
warranty NOT cover" on its own?)


## Part 2: Embedding Index (8 pts)

Our knowledge base = a 12-entry company FAQ **plus** the policy chunks from Part 1.
We embed every piece into a 384-dimensional vector and load them into a **FAISS**
index (`IndexFlatL2` = exact nearest-neighbor search by L2 distance — brute force,
perfect at this scale).


In [ ]:
# Company FAQ knowledge base
documents = [
    "Our company was founded in 2010 and is headquartered in San Francisco.",
    "We offer a 30-day money-back guarantee on all products.",
    "Customer support is available 24/7 via chat, email, or phone.",
    "Shipping is free for orders over $50 within the United States.",
    "Our premium subscription costs $29.99 per month and includes priority support.",
    "Returns must be initiated within 14 days of delivery.",
    "We accept Visa, Mastercard, American Express, and PayPal.",
    "International shipping is available to over 50 countries.",
    "Our mobile app is available for iOS and Android devices.",
    "Enterprise customers receive dedicated account managers.",
    "The warranty period for electronics is 2 years from purchase date.",
    "Student discounts of 20% are available with valid ID.",
]

# Combine the FAQ entries with the policy chunks into one corpus
corpus = documents + chunks
print(f"Corpus: {len(documents)} FAQ entries + {len(chunks)} policy chunks = {len(corpus)} pieces")

In [ ]:
corpus_embeddings = embedding_model.encode(corpus)
print(f"Embedding shape: {corpus_embeddings.shape}")
print(f"→ each piece of text is a {corpus_embeddings.shape[1]}-dimensional vector")

dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)          # exact L2-distance search
index.add(corpus_embeddings.astype("float32"))
print(f"✅ FAISS vector store built with {index.ntotal} vectors")

### 🧪 Experiment

- Embed two paraphrases — `embedding_model.encode(["full refund", "money-back guarantee"])` — and compute `np.linalg.norm(a - b)` between them. Now compare against two unrelated sentences. That distance gap is the entire magic of semantic search.
- Append one absurd off-topic document ("The moon is made of green cheese.") to `corpus`, rebuild the index, and see in Part 3 whether any reasonable question ever retrieves it.


## Part 3: Retrieval (10 pts)

Retrieval = embed the question with the **same** model, find the k nearest document
vectors. Lower L2 distance = more similar. This is the make-or-break step of every
RAG system: if the right document isn't in the top-k, the LLM can't answer correctly
no matter how smart it is.


In [ ]:
def retrieve_similar(query, k=3):
    """Retrieve the k most similar corpus pieces for a query."""
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding.astype("float32"), k)
    return [{"document": corpus[idx], "score": float(distances[0][i]), "index": int(idx)}
            for i, idx in enumerate(indices[0])]

results = retrieve_similar("What is your return policy?", k=3)
for i, r in enumerate(results, 1):
    print(f"{i}. (L2={r['score']:.3f}) {r['document'][:90]}")

In [ ]:
# Probe retrieval quality — including questions designed to confuse it.
probe_queries = [
    "How long do I have to send back a laptop?",   # paraphrase, no shared keywords
    "Do you take PayPal?",
    "What does the warranty not cover?",
    "What time do you close on Sundays?",          # NOT in the corpus — watch what happens
]
for q in probe_queries:
    top = retrieve_similar(q, k=2)
    print(f"\nQ: {q}")
    for r in top:
        print(f"   (L2={r['score']:.3f}) {r['document'][:80]}")

**Notice the last query:** the corpus says nothing about store hours, but FAISS
*always* returns the k nearest neighbors — nearest can still be far. Real systems set a
distance threshold. Keep a failure example like this for **Q3**.


### 🧪 Experiment

- Re-run the probes with `k=1` and then `k=10`. At k=1, how often is the single hit the right one? At k=10, how much junk rides along?
- Compare the L2 scores of a good hit ("Do you take PayPal?") vs the store-hours miss. Propose a numeric threshold that would separate them — then find a legitimate question your threshold would wrongly reject.
- Ask the same question with typos ("wat is ur retrun polcy"). Does semantic search shrug it off where keyword search would face-plant?


## Part 4: Generation with RAG (10 pts)

Now the full pipeline: retrieve → build an augmented prompt → generate. flan-t5-base
is instruction-tuned, so "answer using this context" actually works — and because it's
small and knows nothing about our fictional company, the **difference between RAG and
no-RAG will be dramatic** (that's the point of the comparison).


In [ ]:
def rag_answer(question, k=3):
    """Answer a question with retrieval-augmented generation."""
    retrieved = retrieve_similar(question, k)
    context = "\n".join(r["document"] for r in retrieved)
    prompt = (f"Answer the question using only the context below.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}\nAnswer:")
    answer = generator(prompt, max_new_tokens=80)[0]["generated_text"]
    return {"question": question, "context": context,
            "retrieved_docs": retrieved, "answer": answer}

test_questions = [
    "How long do I have to return a product?",
    "What payment methods do you accept?",
    "How much does the premium subscription cost?",
    "Is there a student discount available?",
]
for q in test_questions:
    r = rag_answer(q)
    print(f"\nQ: {q}\nA: {r['answer']}")

In [ ]:
# The comparison that justifies the whole architecture: RAG vs the same LLM alone.
def compare_approaches(question):
    rag = rag_answer(question)
    pure = generator(f"Question: {question}\nAnswer:", max_new_tokens=80)[0]["generated_text"]
    return {"question": question, "rag_answer": rag["answer"], "pure_llm_answer": pure}

for q in ["What is your return policy?",
          "How much does premium cost?",
          "Do you ship internationally?"]:
    r = compare_approaches(q)
    print(f"\nQ: {r['question']}")
    print(f"  [RAG]      {r['rag_answer']}")
    print(f"  [Pure LLM] {r['pure_llm_answer']}")
    print("-" * 60)

### 🧪 Experiment

- Shrink the chunks to `chunk_size=30, overlap=0`, rebuild `corpus` + the FAISS index (re-run the Part 2 cells), and re-ask the four test questions — watch retrieval fall apart on multi-fact questions like the warranty one.
- Ask a question the corpus can't answer ("Who is the CEO?") through BOTH `rag_answer` and the bare generator. Does RAG admit ignorance better than the bare LLM, or does it hallucinate from irrelevant context?
- Bump `k=3` to `k=8` in `rag_answer`. Does more context help, or does the extra noise drown the model? Small models are especially easy to drown.


**Summarize in a markdown cell (2–3 rows are enough):** a small comparison table —
question | RAG answer | pure-LLM answer | which is right? The pure LLM either refuses,
hallucinates a generic policy, or guesses — it has never seen *your* documents. That
gap is the business case for RAG. Save your best example for **Q2**.


In [ ]:
# ── OPTIONAL (not required, $0 stays $0): swap in a frontier model as the generator ──
# ⚠️ Requires creating a free external account + API key — skip freely.
# The retrieval half stays identical; only generation changes. Two free options:
#
# Option 1 — OpenRouter free tier (sign up free at openrouter.ai, no card needed;
#            save the key as Colab secret OPENROUTER_API_KEY):
# from google.colab import userdata
# from openai import OpenAI
# client = OpenAI(base_url="https://openrouter.ai/api/v1",
#                 api_key=userdata.get("OPENROUTER_API_KEY"))
# def rag_answer_frontier(question, k=3):
#     ctx = "\n".join(r["document"] for r in retrieve_similar(question, k))
#     msg = f"Answer using only this context:\n{ctx}\n\nQuestion: {question}"
#     resp = client.chat.completions.create(
#         model="openai/gpt-oss-20b:free",   # free ids rotate — check openrouter.ai/models
#         messages=[{"role": "user", "content": msg}])
#     return resp.choices[0].message.content
#
# Option 2 — Gemini free tier (key from Google AI Studio, save as Colab secret GOOGLE_API_KEY):
# see the course setup guide; same pattern, different client.
print("Optional cell — skip freely, flan-t5 is enough for full credit.")

## Part 5: Questions (6 pts)

Answer **Q1–Q4** (listed in the Submit section below) in a markdown cell here. Use
concrete evidence from your runs: your chunk-size experiments (Q1), your best
RAG-vs-LLM example (Q2), your retrieval failure (Q3), and a deployment sketch —
where do the documents live, what updates the index, what does it cost (Q4).


## 🎮 Your Domain Playground

The demo taught you the METHOD on a fictional company's documents. Now build RAG over
a knowledge base from **a domain you're passionate about** — a world where YOU are the
expert and can instantly tell a right answer from a confident wrong one. Ideas:

- your industry's public FAQ or policy pages
- a game's wiki pages (build rules, item stats, lore)
- your fantasy league's rulebook or your team's bylaws
- a product manual, a gym's class policies, a band's tour FAQ

**Steps:**

1. **Collect 10–15 passages** (a sentence to a paragraph each) from your domain and
   paste them into `my_documents` below. Longer source? Paste it as one string and
   `chunk_text` it — that's what Part 1 was for.


In [ ]:
# Paste 10-15 passages from YOUR domain, then run this cell.
my_documents = [
    # "Passage 1 from your domain...",
    # "Passage 2 from your domain...",
]

# Long source document instead? Uncomment:
# my_long_doc = """(paste the full text here)"""
# my_documents = my_documents + chunk_text(my_long_doc, chunk_size=60, overlap=15)

if my_documents:
    corpus = my_documents
    corpus_embeddings = embedding_model.encode(corpus)
    index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
    index.add(corpus_embeddings.astype("float32"))
    print(f"✅ Rebuilt the vector store over YOUR domain: {index.ntotal} pieces")
else:
    print("⬆️ Add your passages to my_documents, then re-run this cell.")

2. **Re-ask through the same pipeline:** call `rag_answer()` on 4–5 questions a
   real fan/insider/customer of your domain would ask, plus one it can't answer. Run
   `compare_approaches()` on your best one.
3. **The insider test (markdown cell, 4–6 sentences):** as the domain expert, grade the
   answers — right, subtly wrong, or confidently hallucinated? Which questions needed
   bigger chunks or a different k than the demo's settings? An answer that would fool
   an outsider but not you is exactly the failure mode businesses pay experts to catch —
   name one if you found it.


## 🤔 Debrief

1. Your FAISS search *always* returns k results, relevant or not — and the LLM will
   confidently answer from bad context. In production, what guardrails stop a customer
   seeing an answer built on garbage retrieval?
2. When is RAG the WRONG tool? (Think: data that changes every second, questions
   needing reasoning across ALL documents at once, or 20 documents you could just
   paste into one long-context prompt.)


## 📤 Submit (HW7 — 40 points)

**Deliverables (submit through Canvas):**
- Completed `.ipynb` executed top-to-bottom, INCLUDING the 🎮 Playground run on documents from your own domain (built yourself — no NotebookLM or other pre-built RAG services)
- A short RAG vs pure-LLM comparison table (markdown cell)
- Your insider-test writeup grading the answers on your domain's questions

**Answer these questions in a markdown cell at the end of your notebook:**
- **Q1:** How did you decide on chunk size and overlap?
- **Q2:** Show an example where RAG improved the LLM's answer.
- **Q3:** Show an example where retrieval failed. Why?
- **Q4:** How would you deploy this for a real business application?

**Grading:**

| Part | Points |
|---|---|
| Part 1: Document Chunking | 6 |
| Part 2: Embedding Index | 8 |
| Part 3: Retrieval | 10 |
| Part 4: Generation with RAG | 10 |
| Part 5: Questions | 6 |

**Due Sunday 10:00 p.m.** — see the Canvas assignment for the exact date. Late work is not accepted (see syllabus).

*AI policy: AI tools are allowed and encouraged. Add one line acknowledging what you used, and be ready to explain every part of what you submit.*
